In [28]:
# Import des liberairies

#----Python
import os

# ----- Spark
from pyspark.sql.session import SparkSession
import pyspark.sql.functions as sf
from pyspark.ml.feature import StringIndexer, VectorAssembler, OneHotEncoder
from pyspark.ml.pipeline import Pipeline
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

#----Autres-------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- --Initialisation Spark ---
spark = (
    SparkSession.builder.appName('Projet')
    .config('spark.driver.memory', '4g') # autoriser le driver d'utiliser jusqu'à 4gb
    .config('spark.executer.memory','4g')
    .config('spark.sql.shuffle.partitions','50') # par defaut c'est 200, 50 pour réduire la tension sur la machine
    .config('spark.sql.execution.arrow.pyspark.enabaled' ,'true') # pour une transition rapide entre pandas et spark df
    .getOrCreate()
)

# --- Formatage du texte ---
# character Gras
B = '\033[1m' 
BE = '\033[0m'

#Chemin vers les données sur ma machine
CHEMIN_INTERIM = '../data/interim'

In [30]:
df_LabelConf = spark.read.parquet(CHEMIN_INTERIM + '/df_LabelConf.parquet')
df_LabelConf.printSchema()

root
 |-- CdSta: string (nullable = true)
 |-- NmNoSta: string (nullable = true)
 |-- DtPrel: string (nullable = true)
 |-- AnPrel: string (nullable = true)
 |-- MoPrel: string (nullable = true)
 |-- NoDep: string (nullable = true)
 |-- CdTpSta: string (nullable = true)
 |-- LbTpSta: string (nullable = true)
 |-- CapNomSta: double (nullable = true)
 |-- A3_Temperature: double (nullable = true)
 |-- A3_pH: double (nullable = true)
 |-- A3_MES: double (nullable = true)
 |-- A3_DBO5: double (nullable = true)
 |-- A3_DCO: double (nullable = true)
 |-- A3_NTK: double (nullable = true)
 |-- A3_NH4: double (nullable = true)
 |-- A3_NO2: double (nullable = true)
 |-- A3_NO3: double (nullable = true)
 |-- A3_PT: double (nullable = true)
 |-- A3_Nglobal: double (nullable = true)
 |-- A3_Volume: double (nullable = true)
 |-- A3_Pluie: double (nullable = true)
 |-- A4_Temperature: double (nullable = true)
 |-- A4_pH: double (nullable = true)
 |-- A4_MES: double (nullable = true)
 |-- A4_DBO5: doub

In [32]:
# On filtre les lignes où on a les entrées et la sorties de rendemment (pour avoir un rendement valide)
df_PreReg = (
    df_LabelConf
    .select(  
        
        # Identifiants (pas pour le modèle, mais pratiques pour debug/analyse)
        'CdSta'
        ,'DtPrel'
        
        # Features catégorielles (à encoder ensuite)
        ,'CatSta'
        ,'NoDep'
        ,'MoPrel'
        ,'AnPrel'
        
        # Features numériques (entrée + station)
        ,'A3_DBO5'
        ,'A3_DCO'
        ,'A3_MES'
        ,'A3_NTK'
        ,'A3_NH4'
        ,'A3_Nglobal'
        ,'CapNomSta'
        
        # Cible
        ,'Rdt_DBO5'
        ,'Rdt_DCO'
        ,'Rdt_MES'
        ,'Rdt_NTK'
        ,'Rdt_NH4'
        ,'Rdt_Nglobal'
    )
    .filter(
        sf.col('Rdt_DBO5').isNotNull()
        & sf.col('Rdt_DCO').isNotNull()
        & sf.col('Rdt_MES').isNotNull()
        & sf.col('Rdt_NTK').isNotNull()
        & sf.col('Rdt_NH4').isNotNull()
        & sf.col('Rdt_Nglobal').isNotNull()
        
        & sf.col('A3_DBO5').isNotNull()
        & sf.col('A3_DCO').isNotNull()
        & sf.col('A3_MES').isNotNull()
        & sf.col('A3_NTK').isNotNull()
        & sf.col('A3_NH4').isNotNull()
        & sf.col('A3_Nglobal').isNotNull()
        & sf.col('CapNomSta').isNotNull()
    )
)
print(df_PreReg.count())
df_PreReg.select('Rdt_DBO5','Rdt_DCO','Rdt_MES','Rdt_NTK','Rdt_NH4','Rdt_Nglobal').describe().show()


28923
+-------+-----------------+-----------------+-----------------+------------------+------------------+------------------+
|summary|         Rdt_DBO5|          Rdt_DCO|          Rdt_MES|           Rdt_NTK|           Rdt_NH4|       Rdt_Nglobal|
+-------+-----------------+-----------------+-----------------+------------------+------------------+------------------+
|  count|            28923|            28923|            28923|             28923|             28923|             28923|
|   mean|97.53625811607313|94.63774958109173|96.54822476119283| 93.24627301179981| 93.91641720735029| 83.98366162866962|
| stddev|5.656127528322749|6.908454418588321|9.358897516569241|11.468917347242161|14.664918685395769|17.313480917976335|
|    min|           -100.0|           -100.0|           -100.0|            -100.0|            -100.0|            -100.0|
|    max|            100.0|99.92663656884876|            100.0|             100.0|             100.0|             100.0|
+-------+-----------------

### String Indexing & One-Hot Encoding

In [34]:
col_idx = ['AnPrel', 'MoPrel', 'NoDep','CatSta']

# Indexing 
indexeur = [StringIndexer(inputCol = c, outputCol = c + '_idx',handleInvalid="keep") for c in col_idx]

# OHE
encodeur= [OneHotEncoder(inputCol = c + '_idx', outputCol = c + '_ohe',handleInvalid="keep") for c in col_idx]

# df_PreRegEncod
df_PreRegEncod = Pipeline(stages=[*indexeur,*encodeur]).fit(df_PreReg).transform(df_PreReg)
df_PreRegEncod.printSchema()

root
 |-- CdSta: string (nullable = true)
 |-- DtPrel: string (nullable = true)
 |-- CatSta: string (nullable = true)
 |-- NoDep: string (nullable = true)
 |-- MoPrel: string (nullable = true)
 |-- AnPrel: string (nullable = true)
 |-- A3_DBO5: double (nullable = true)
 |-- A3_DCO: double (nullable = true)
 |-- A3_MES: double (nullable = true)
 |-- A3_NTK: double (nullable = true)
 |-- A3_NH4: double (nullable = true)
 |-- A3_Nglobal: double (nullable = true)
 |-- CapNomSta: double (nullable = true)
 |-- Rdt_DBO5: double (nullable = true)
 |-- Rdt_DCO: double (nullable = true)
 |-- Rdt_MES: double (nullable = true)
 |-- Rdt_NTK: double (nullable = true)
 |-- Rdt_NH4: double (nullable = true)
 |-- Rdt_Nglobal: double (nullable = true)
 |-- AnPrel_idx: double (nullable = false)
 |-- MoPrel_idx: double (nullable = false)
 |-- NoDep_idx: double (nullable = false)
 |-- CatSta_idx: double (nullable = false)
 |-- AnPrel_ohe: vector (nullable = true)
 |-- MoPrel_ohe: vector (nullable = true)
 

In [35]:
# Features finales
col_feat = ['AnPrel_ohe', 'MoPrel_ohe', 'NoDep_ohe', 'CatSta_ohe','CapNomSta', 'A3_DBO5', 'A3_DCO', 'A3_MES','A3_NTK','A3_NH4','A3_Nglobal']

# Assembleur : création de la colonne feature
assembleur = VectorAssembler(inputCols = col_feat, outputCol= 'features')

# df_Reg
df_Reg = (
    Pipeline(stages=[assembleur])
    .fit(df_PreRegEncod)
    .transform(df_PreRegEncod)
    .select(  
        # Identifiants (pas pour le modèle, mais pratiques pour debug/analyse)
        'CdSta'
        ,'DtPrel'
        
        # Features catégorielles (à encoder ensuite)
        ,'CatSta'
        ,'NoDep'
        ,'MoPrel'
        ,'AnPrel'
        
        # Features numériques (entrée + station)
        ,'A3_DBO5'
        ,'A3_DCO'
        ,'A3_MES'
        ,'A3_NTK'
        ,'A3_NH4'
        ,'A3_Nglobal'
        ,'CapNomSta'
        
        # Cible
        ,'Rdt_DBO5'
        ,'Rdt_DCO'
        ,'Rdt_MES'
        ,'Rdt_NTK'
        ,'Rdt_NH4'
        
        # feature
        ,'features'
    ))
df_Reg.printSchema()

root
 |-- CdSta: string (nullable = true)
 |-- DtPrel: string (nullable = true)
 |-- CatSta: string (nullable = true)
 |-- NoDep: string (nullable = true)
 |-- MoPrel: string (nullable = true)
 |-- AnPrel: string (nullable = true)
 |-- A3_DBO5: double (nullable = true)
 |-- A3_DCO: double (nullable = true)
 |-- A3_MES: double (nullable = true)
 |-- A3_NTK: double (nullable = true)
 |-- A3_NH4: double (nullable = true)
 |-- A3_Nglobal: double (nullable = true)
 |-- CapNomSta: double (nullable = true)
 |-- Rdt_DBO5: double (nullable = true)
 |-- Rdt_DCO: double (nullable = true)
 |-- Rdt_MES: double (nullable = true)
 |-- Rdt_NTK: double (nullable = true)
 |-- Rdt_NH4: double (nullable = true)
 |-- features: vector (nullable = true)



In [36]:
# creation de données d'entrainement et test data
df_train, df_test = df_Reg.randomSplit([.8,.2],seed=2026)

In [61]:
def eval_metrics(reg_method, df_predictions, colCibl,colPred):
    print(f'Evalution des métriques {reg_method} pour {colCibl} :')
    for metric in ['rmse','r2','mae','var']:
        evaluator = RegressionEvaluator(predictionCol = colPred ,labelCol = colCibl ,metricName = metric)
        eval_value = round(evaluator.evaluate(df_predictions),2)
        print(f'{B}{metric}{BE}:{eval_value}')

### Random Forest

In [49]:
col_cibles =['Rdt_DBO5', 'Rdt_DCO', 'Rdt_MES', 'Rdt_NTK', 'Rdt_NH4']
# col_cibles =['Rdt_DBO5', 'Rdt_DCO']

# Feature imporantces 

attrs = df_train.schema['features'].metadata['ml_attr']['attrs']

feat_names = [None] * df_train.first()['features'].size
for type_attr in ['numeric', 'binary', 'nominal']:
    if type_attr in attrs:
        for a in attrs[type_attr]:
            feat_names[a['idx']] = a['name']

In [53]:
# Model RF
for colCibl in col_cibles:
    # Regression lineaire: parametère par default 
    rf = RandomForestRegressor(featuresCol='features',labelCol= colCibl,seed=2026)
    # Default RandomForest DF
    df_rf_pred_NonParam = rf.fit(df_train).transform(df_test)
    
    # Grille des parametères 
    rf_grilPram = (
        ParamGridBuilder()
        .addGrid(rf.numTrees, [20,50])
        .addGrid(rf.maxDepth, [10,15])
        # .addGrid(rf.minInstancesPerNode, [1, 5])
        .build()
    )
    # Evaluateur du model regression lineaire 
    rf_evaluateur = RegressionEvaluator(predictionCol = 'prediction' ,labelCol = colCibl ,metricName = 'rmse')
    
    # Validateur croisé des parameters du model
    rf_validateur = CrossValidator(
        estimator = rf
        ,estimatorParamMaps = rf_grilPram
        ,evaluator = rf_evaluateur
        ,numFolds=2
        ,parallelism=4
        ,seed=2026
    )
    
    # Modele du Validateur
    rf_validateur_model = rf_validateur.fit(df_train) # 3x3x2 = 18 models tested
    
    Meil_numTrees =              rf_validateur_model.bestModel.getNumTrees
    Meil_maxDepth =              rf_validateur_model.bestModel.getMaxDepth()
    Meil_minInstancesPerNode =   rf_validateur_model.bestModel.getMinInstancesPerNode()
    
    # Prediction sur le df_test
    df_rf_pred = rf_validateur_model.bestModel.transform(df_test)
    
    
    # Evalutation du modèle et Affichage
    print(f'\n{B}Modele Random Forest (RF){BE} de {B}{colCibl} :{BE}')
    print(f'Parametres du meilleur modèle:')
    print(f'{B}NumTrees{BE} : {Meil_numTrees}')
    print(f'{B}MaxDepthMeil_maxDepth{BE} : {Meil_maxDepth}\n')
    
    # print(f'{B}Meilleur MinInstancesPerNode :{BE} {Meil_minInstancesPerNode}\n')
    eval_metrics('RF - Sans Parametrage', df_rf_pred_NonParam, colCibl)
    eval_metrics('RF - Avec les meilleures paramètres', df_rf_pred, colCibl)
    
    # Affichage avec importances
    imp_array = rf_validateur_model.bestModel.featureImportances.toArray()
    
    print(f'\nImportance des Features:')
    for nom, imp in sorted(zip(feat_names, imp_array), key=lambda x: -x[1])[:15]:
        print(f"{B}{nom:<35}{BE} {imp:.4f}")
    print(f'-------------------------------------------\n-------------------------------------------')
    
    # conclusion le tuning ne change rien 


 Modele Random Forest (RF) de Rdt_DBO5 :
Parametres du meilleur modèle:
NumTrees : 50
MaxDepthMeil_maxDepth : 10

Evalution des métriques RF - Sans Parametrage pour Rdt_DBO5 :
rmse:5.43
r2:0.19
mae:1.41
var:5.59
Evalution des métriques RF - Avec les meilleures paramètres pour Rdt_DBO5 :
rmse:5.39
r2:0.2
mae:1.3
var:9.31

Importance des Features:
A3_DBO5                             0.1291
CapNomSta                           0.1071
A3_DCO                              0.0983
A3_Nglobal                          0.0729
A3_MES                              0.0714
A3_NTK                              0.0711
A3_NH4                              0.0658
NoDep_ohe_27                        0.0364
MoPrel_ohe_04                       0.0328
AnPrel_ohe_2021                     0.0298
MoPrel_ohe_12                       0.0237
MoPrel_ohe_05                       0.0221
NoDep_ohe_77                        0.0217
MoPrel_ohe_06                       0.0211
CatSta_ohe_petite                   0.0203

 Mode

In [59]:
# 1. Create a list of stages (one RF model per target)
stages = []
for colCibl in col_cibles:
    # We give each model a unique prediction column name
    rf = RandomForestRegressor(
        featuresCol='features', 
        labelCol=colCibl, 
        predictionCol=f"pred_{colCibl}", 
        numTrees=50, 
        maxDepth=10, 
        seed=2026
    )
    stages.append(rf)

# 2. Wrap everything into a single Pipeline
pipeline = Pipeline(stages=stages)

# 3. Fit the model once
# This trains all targets in sequence (or parallel if your cluster allows)
multi_target_model = pipeline.fit(df_train)

# 4. Predict everything at once
df_predictions = multi_target_model.transform(df_test)

# Now df_predictions contains: pred_Rdt_DBO5, pred_Rdt_DCO, etc.
df_predictions.select('features',col_cibles, *[f"pred_{c}" for c in col_cibles]).show()

+--------------------+-----------------+-----------------+-----------------+-----------------+-----------------+
|            features|    pred_Rdt_DBO5|     pred_Rdt_DCO|     pred_Rdt_MES|     pred_Rdt_NTK|     pred_Rdt_NH4|
+--------------------+-----------------+-----------------+-----------------+-----------------+-----------------+
|(54,[1,4,27,44,47...|98.91582964074216|96.77079171245849| 98.7498727717331|94.53268838947531|96.24172984922814|
|(54,[0,10,27,45,4...|94.58399289541813|88.67030790229072|90.30658994003117|72.50986460167708|65.28340821876435|
|(54,[1,7,27,44,47...|98.38647757211183|  95.345945626943| 94.3281537839411| 94.4810496388625|95.35181183148148|
|(54,[0,12,27,43,4...|98.48530283667402|96.36489034633387|98.59666680956154|95.50901299583991|96.51676258253819|
|(54,[1,13,27,43,4...| 99.0674885187896|97.14054215786614|98.81630736461305|96.75566287686283|97.64923875486518|
|(54,[0,3,27,43,47...|98.16576982942591|94.91056798000338|97.96064765088875| 95.5609970720729|96

In [89]:
# Now df_predictions contains: pred_Rdt_DBO5, pred_Rdt_DCO, etc.
df_predictions.select( *[c for c in col_cibles], *[f"pred_{c}" for c in col_cibles]).show(truncate=10)

+----------+----------+----------+----------+----------+-------------+------------+------------+------------+------------+
|  Rdt_DBO5|   Rdt_DCO|   Rdt_MES|   Rdt_NTK|   Rdt_NH4|pred_Rdt_DBO5|pred_Rdt_DCO|pred_Rdt_MES|pred_Rdt_NTK|pred_Rdt_NH4|
+----------+----------+----------+----------+----------+-------------+------------+------------+------------+------------+
|99.1176...|96.0945...|98.7916...|96.3865...|98.0851...|   98.9158...|  96.7707...|  98.7498...|  94.5326...|  96.2417...|
|98.9285...|93.2011...|      98.8|85.4368...|84.5238...|   94.5839...|  88.6703...|  90.3065...|  72.5098...|  65.2834...|
|     96.76|92.4242...|  89.84375|98.0222...|97.7349...|   98.3864...|  95.3459...|  94.3281...|  94.4810...|  95.3518...|
|      98.8|96.7362...|99.1451...|97.0489...|99.2337...|   98.4853...|  96.3648...|  98.5966...|  95.5090...|  96.5167...|
|99.2857...|97.2616...|99.5192...|98.0219...|99.2857...|   99.0674...|  97.1405...|  98.8163...|  96.7556...|  97.6492...|
|      96.5|93.6

In [67]:
    # Affichage
    print(f'\n{B}Modele Random Forest (RF){BE} de {B}{colCibl} :{BE}')
    for colCibl in col_cibles:
        eval_metrics('RF', df_predictions, colCibl, f"pred_{colCibl}")

    print(f'-------------------------------------------\n-------------------------------------------')


Modele Random Forest (RF) de Rdt_NH4 :
Evalution des métriques RF pour Rdt_DBO5 :
rmse:5.39
r2:0.2
mae:1.3
var:9.31
Evalution des métriques RF pour Rdt_DCO :
rmse:5.9
r2:0.26
mae:2.06
var:13.48
Evalution des métriques RF pour Rdt_MES :
rmse:9.39
r2:0.18
mae:2.66
var:16.26
Evalution des métriques RF pour Rdt_NTK :
rmse:10.38
r2:0.23
mae:4.66
var:28.28
Evalution des métriques RF pour Rdt_NH4 :
rmse:12.72
r2:0.24
mae:5.75
var:45.34
-------------------------------------------
-------------------------------------------
